# TP Python — Rétropropagation et méthodes d'apprentissage

*Séance en salle informatique. Ce TP suppose connus les perceptrons et les réseaux du TP [TP Python : programmer perceptrons et réseaux](https://openyourmath.org/exercise/96ip) (`96ip`).*

**Au programme**
1. programmer la propagation avant et la perte ;
2. programmer la **rétropropagation**, et la vérifier par différences finies et sur l'exercice [Une rétropropagation de bout en bout](https://openyourmath.org/exercise/IVdd) (`IVdd`) ;
3. entraîner un réseau sur le XOR et observer ce qui peut mal se passer (pas, initialisation, ReLU morte) ;
4. comparer les **méthodes d'apprentissage** : lot complet, mini-lots, un exemple à la fois ;
5. mesurer la **généralisation** : jeu d'entraînement et de validation, sur-apprentissage, pénalisation $L^2$.

**Conventions** (celles du poly)
- Un lot de $m$ exemples est un tableau `X` de forme `(n_0, m)`, les cibles un tableau `T` de forme `(n_L, m)`.
- Couche $\ell$ : $z^{[\ell]}=W^{[\ell]}a^{[\ell-1]}+b^{[\ell]}$ et $a^{[\ell]}=\varphi^{[\ell]}(z^{[\ell]})$, avec $a^{[0]}=x$.
- **Perte moyenne** sur un lot de $m$ exemples : $E=\dfrac1m\sum_{i=1}^m \dfrac12\|a^{[L]}_i-t_i\|^2$.
- Le dernier lot d'une époque peut être plus petit : la moyenne est toujours prise sur la **taille réelle** du lot.

Les cellules `À COMPLÉTER` sont à écrire ; les cellules de test qui suivent vous disent si c'est juste.

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (5, 4)


def sigmoide(z):
    z = np.asarray(z, dtype=float)
    e = np.exp(-np.abs(z))
    return np.where(z >= 0, 1 / (1 + e), e / (1 + e))


def d_sigmoide(z):
    s = sigmoide(z)
    return s * (1 - s)


def relu(z):
    return np.maximum(0, z)


def d_relu(z):
    return (z > 0).astype(float)   # convention : ReLU'(0) = 0


def identite(z):
    return z


def d_identite(z):
    return np.ones_like(z)


# Chaque activation est donnée avec sa dérivée
ACT = {"sigmoide": (sigmoide, d_sigmoide), "relu": (relu, d_relu), "identite": (identite, d_identite)}

## 1. Le réseau, sa propagation avant et sa perte

Un réseau est décrit par une liste `params = [[W1, b1], [W2, b2], ...]` et par la liste `acts` des noms de ses activations (`"relu"`, `"sigmoide"`, `"identite"`). `init_reseau` (fournie) tire les poids au hasard.

Écrivez `avant`, qui renvoie **toutes** les préactivations $z^{[\ell]}$ et toutes les sorties $a^{[\ell]}$ (on en aura besoin pour la rétropropagation), puis `perte`.

In [ ]:
def init_reseau(tailles, graine=0, echelle=0.5):
    """tailles = [n0, n1, ..., nL]. Poids gaussiens d'écart-type `echelle`, biais nuls."""
    rng = np.random.default_rng(graine)
    return [[rng.normal(0, echelle, (n, p)), np.zeros(n)] for p, n in zip(tailles[:-1], tailles[1:])]

In [ ]:
def avant(X, params, acts):
    """Propagation avant. Renvoie Z = [z1, ..., zL] et A = [a0 = X, a1, ..., aL]."""
    A = [X]
    Z = []
    for (W, b), nom in zip(params, acts):
        # À COMPLÉTER : préactivation z = W a + b (le biais s'ajoute à chaque colonne), puis activation
        raise NotImplementedError
    return Z, A


def perte(sortie, T):
    """Perte moyenne : (1/m) * somme sur les exemples de (1/2) * ||sortie_i - T_i||^2."""
    # À COMPLÉTER
    raise NotImplementedError

**Test : le réseau de l'exercice [Une rétropropagation de bout en bout](https://openyourmath.org/exercise/IVdd) (`IVdd`).** Un neurone caché ReLU puis une sortie linéaire, $h=\mathrm{ReLU}(w_1x+b_1)$, $F=w_2h+b_2$, avec $(w_1,b_1,w_2,b_2)=(1,0,2,0)$, la donnée $x=1$, $t=1$, et une perte $\tfrac12(F-t)^2$. L'exercice donne $F=2$ et $E=\tfrac12$.

In [ ]:
# Test
params_td = [[np.array([[1.0]]), np.array([0.0])], [np.array([[2.0]]), np.array([0.0])]]
acts_td = ["relu", "identite"]
X_td = np.array([[1.0]])
T_td = np.array([[1.0]])

Z, A = avant(X_td, params_td, acts_td)
assert len(Z) == 2 and len(A) == 3, "Z contient L tableaux, A en contient L + 1"
assert np.isclose(A[-1][0, 0], 2.0), "F(1) doit valoir 2"
assert np.isclose(perte(A[-1], T_td), 0.5), "la perte initiale doit valoir 1/2"

# un lot de deux exemples : la perte est une moyenne
assert np.isclose(perte(np.array([[1.0, 3.0]]), np.array([[0.0, 0.0]])), 0.5 * (1 + 9) / 2)
print("Propagation avant et perte : OK")

## 2. La rétropropagation

On note $\delta^{[\ell]}=\dfrac{\partial E}{\partial z^{[\ell]}}$ (un tableau de même forme que $z^{[\ell]}$). Les règles du cours se résument à :

$$\delta^{[L]}=\frac1m\,(a^{[L]}-t)\odot\varphi^{[L]\prime}(z^{[L]}),\qquad
\delta^{[\ell-1]}=\big((W^{[\ell]})^{\top}\delta^{[\ell]}\big)\odot\varphi^{[\ell-1]\prime}(z^{[\ell-1]}),$$
$$\frac{\partial E}{\partial W^{[\ell]}}=\delta^{[\ell]}\,(a^{[\ell-1]})^{\top},\qquad
\frac{\partial E}{\partial b^{[\ell]}}=\text{somme des colonnes de }\delta^{[\ell]}.$$

où $\odot$ est le produit terme à terme. Le facteur $\frac1m$ vient de la perte moyenne.

**Attention aux indices.** Python numérote à partir de 0 : `Z[l]` est $z^{[l+1]}$, `A[l]` est $a^{[l]}$, `params[l]` contient $W^{[l+1]}$ et $b^{[l+1]}$ et `acts[l]` est l'activation de la couche $l+1$.

Écrivez `arriere`, qui renvoie la liste des gradients `[(dW1, db1), (dW2, db2), ...]`.

In [ ]:
def arriere(Z, A, params, acts, T):
    """Rétropropagation. Renvoie grads = [(dW1, db1), ..., (dWL, dbL)]."""
    m = T.shape[1]
    L = len(params)
    grads = [None] * L
    delta = None  # À COMPLÉTER : delta de la dernière couche
    for l in range(L - 1, -1, -1):
        W, b = params[l]
        # À COMPLÉTER : gradients (dW, db) de la couche l, rangés dans grads[l],
        # puis delta de la couche précédente (seulement si l > 0)
        raise NotImplementedError
    return grads

**Test 1 : les gradients de l'exercice [Une rétropropagation de bout en bout](https://openyourmath.org/exercise/IVdd) (`IVdd`).** L'exercice donne $\dfrac{\partial E}{\partial w_1}=2$, $\dfrac{\partial E}{\partial b_1}=2$, $\dfrac{\partial E}{\partial w_2}=1$, $\dfrac{\partial E}{\partial b_2}=1$.

In [ ]:
# Test 1
Z, A = avant(X_td, params_td, acts_td)
(dW1, db1), (dW2, db2) = arriere(Z, A, params_td, acts_td, T_td)
assert np.allclose([dW1[0, 0], db1[0], dW2[0, 0], db2[0]], [2.0, 2.0, 1.0, 1.0]), "gradients de l'exercice IVdd : (2, 2, 1, 1)"
print("Gradients de l'exercice IVdd : OK")

**Test 2 : la vérification par différences finies** (fournie). Pour chaque paramètre $\theta$, on compare le gradient de `arriere` à $\dfrac{E(\theta+h)-E(\theta-h)}{2h}$. C'est le test que l'on fait **toujours** avant de se fier à une rétropropagation.

In [ ]:
def verifier_gradient(params, acts, X, T, h=1e-6):
    """Plus grand écart relatif entre le gradient de `arriere` et les différences finies centrées."""
    Z, A = avant(X, params, acts)
    grads = arriere(Z, A, params, acts, T)
    pire = 0.0
    for (W, b), (dW, db) in zip(params, grads):
        for tableau, g in ((W, dW), (b, db)):
            for idx in np.ndindex(tableau.shape):
                ancien = tableau[idx]
                tableau[idx] = ancien + h
                E_plus = perte(avant(X, params, acts)[1][-1], T)
                tableau[idx] = ancien - h
                E_moins = perte(avant(X, params, acts)[1][-1], T)
                tableau[idx] = ancien
                num = (E_plus - E_moins) / (2 * h)
                pire = max(pire, abs(num - g[idx]) / max(1e-8, abs(num) + abs(g[idx])))
    return pire


rng = np.random.default_rng(1)
for tailles, acts_test in [([3, 4, 2], ["sigmoide", "sigmoide"]),
                           ([3, 5, 4, 2], ["relu", "relu", "identite"]),
                           ([3, 4, 1], ["relu", "sigmoide"])]:
    P = init_reseau(tailles, graine=2, echelle=1.0)
    Xr = rng.normal(size=(tailles[0], 6))
    Tr = rng.uniform(size=(tailles[-1], 6))
    ecart = verifier_gradient(P, acts_test, Xr, Tr)
    assert ecart < 1e-5, f"réseau {tailles} : écart relatif {ecart:.1e}"
    print(f"réseau {tailles} : écart relatif maximal {ecart:.1e}")
print("Rétropropagation : OK")

## 3. Mise à jour des paramètres et entraînement

La descente de gradient met à jour **tous** les paramètres avec le gradient calculé aux paramètres courants : $\theta\leftarrow\theta-\eta\,\dfrac{\partial E}{\partial\theta}$.
Écrivez `mise_a_jour(params, grads, eta)`, qui modifie `params` **en place**.

In [ ]:
def mise_a_jour(params, grads, eta):
    """Descente de gradient : modifie params en place."""
    for (W, b), (dW, db) in zip(params, grads):
        # À COMPLÉTER
        raise NotImplementedError

**Test : un pas de l'exercice [Une rétropropagation de bout en bout](https://openyourmath.org/exercise/IVdd) (`IVdd`).** Avec $\eta=0{,}1$, l'exercice donne les nouveaux paramètres $(0{,}8\,;-0{,}2\,;1{,}9\,;-0{,}1)$ et une perte qui passe de $0{,}5$ à $0{,}0008$.

In [ ]:
# Test
Z, A = avant(X_td, params_td, acts_td)
grads = arriere(Z, A, params_td, acts_td, T_td)
mise_a_jour(params_td, grads, 0.1)
nouveaux = [params_td[0][0][0, 0], params_td[0][1][0], params_td[1][0][0, 0], params_td[1][1][0]]
assert np.allclose(nouveaux, [0.8, -0.2, 1.9, -0.1]), nouveaux
assert np.isclose(perte(avant(X_td, params_td, acts_td)[1][-1], T_td), 0.0008)
print("Un pas de descente : OK, la perte passe de 0,5 à 0,0008")

La fonction `entrainer` (fournie) répète ces étapes. Elle découpe les données en **lots** de taille `taille_lot` (`None` : tout le jeu d'un coup ; `1` : un exemple à la fois), mélange les exemples à chaque époque, et note la perte à la fin de chaque époque. Elle accepte aussi un jeu de validation et un paramètre `lam` de pénalisation, utilisés plus loin.

In [ ]:
def entrainer(X, T, params, acts, eta=0.5, epoques=100, taille_lot=None, graine=0,
              X_val=None, T_val=None, lam=0.0):
    """Entraîne `params` (en place). Renvoie {"train": [...], "val": [...]} (perte à chaque époque)."""
    rng = np.random.default_rng(graine)
    m = X.shape[1]
    taille = m if taille_lot is None else taille_lot
    historique = {"train": [], "val": []}
    for _ in range(epoques):
        ordre = rng.permutation(m)
        for debut in range(0, m, taille):
            idx = ordre[debut:debut + taille]
            Z, A = avant(X[:, idx], params, acts)
            grads = arriere(Z, A, params, acts, T[:, idx])
            if lam > 0:
                grads = ajouter_decroissance(grads, params, lam)
            mise_a_jour(params, grads, eta)
        historique["train"].append(perte(avant(X, params, acts)[1][-1], T))
        if X_val is not None:
            historique["val"].append(perte(avant(X_val, params, acts)[1][-1], T_val))
    return historique

## 4. Le XOR : entraîner un réseau, et ce qui peut mal se passer

Au TP [TP Python : programmer perceptrons et réseaux](https://openyourmath.org/exercise/96ip) (`96ip`), on avait trouvé les poids du XOR à la main. Cette fois, le réseau les **apprend** : deux entrées, trois neurones cachés sigmoïdes, une sortie sigmoïde.

In [ ]:
X_xor = np.array([[0.0, 1.0, 0.0, 1.0],
                  [0.0, 0.0, 1.0, 1.0]])
T_xor = np.array([[0.0, 1.0, 1.0, 0.0]])
acts_xor = ["sigmoide", "sigmoide"]

params = init_reseau([2, 3, 1], graine=0, echelle=1.0)
hist = entrainer(X_xor, T_xor, params, acts_xor, eta=2.0, epoques=3000)
sortie = avant(X_xor, params, acts_xor)[1][-1]
assert hist["train"][-1] < 0.01, "le réseau doit apprendre le XOR"
print("Sorties après apprentissage :", sortie.round(3), "pour les cibles", T_xor)
plt.semilogy(hist["train"])
plt.xlabel("époque")
plt.ylabel("perte")
plt.title("Apprentissage du XOR")
plt.show()

### 4.a. Le pas d'apprentissage $\eta$

**Prédiction avant d'exécuter.** Entraînez le même réseau (même graine) avec $\eta\in\{0{,}05;\ 0{,}5;\ 2;\ 10\}$ pendant 2000 époques. Que va-t-il se passer pour la perte ?

In [ ]:
etas = [0.05, 0.5, 2.0, 10.0]
for eta in etas:
    params = init_reseau([2, 3, 1], graine=0, echelle=1.0)
    # À COMPLÉTER : appeler entrainer (2000 époques, pas eta) et ranger le résultat dans hist
    hist = None
    plt.semilogy(hist["train"], label=f"η = {eta}")
plt.xlabel("époque")
plt.ylabel("perte")
plt.legend()
plt.show()

**Question.** Que constatez-vous pour les petits pas ? Quelle est la valeur du « plateau » sur lequel la perte stagne, et pourquoi ?

**Votre réponse :** *(à rédiger ici)*

### 4.b. L'initialisation

On entraîne le même réseau de trois façons : poids tirés au hasard (référence), **tous les poids nuls**, et **tous les poids égaux à 0,5**.

**Prédiction.** Que va-t-il se passer dans les deux derniers cas ?

In [ ]:
def init_constante(tailles, c):
    """Réseau dont tous les poids valent c et tous les biais 0."""
    # À COMPLÉTER : même structure que init_reseau, avec np.full
    raise NotImplementedError

In [ ]:
# Test
P = init_constante([2, 3, 1], 0.5)
assert P[0][0].shape == (3, 2) and P[1][0].shape == (1, 3) and np.all(P[0][0] == 0.5) and np.all(P[1][1] == 0)

resultats = {}
for nom, depart in [("hasard", init_reseau([2, 3, 1], graine=0, echelle=1.0)),
                    ("zéro", init_constante([2, 3, 1], 0.0)),
                    ("constante 0,5", init_constante([2, 3, 1], 0.5))]:
    h = entrainer(X_xor, T_xor, depart, acts_xor, eta=2.0, epoques=3000)
    resultats[nom] = (h, depart)
    lignes_egales = all(np.allclose(depart[0][0][0], depart[0][0][j]) for j in range(1, 3))
    print(f"{nom:14}: perte finale {h['train'][-1]:.4f} | lignes de W1 identiques : {lignes_egales}")

assert resultats["hasard"][0]["train"][-1] < 0.01
assert resultats["zéro"][0]["train"][-1] > 0.1 and resultats["constante 0,5"][0]["train"][-1] > 0.05

# Pour l'initialisation nulle, les gradients sont nuls dès le départ :
P0 = init_constante([2, 3, 1], 0.0)
Z, A = avant(X_xor, P0, acts_xor)
print("Gradients au départ (poids nuls) :", [np.abs(g[0]).max() for g in arriere(Z, A, P0, acts_xor, T_xor)])

**Question.** Expliquez les deux échecs. Pourquoi les poids nuls ne bougent-ils pas du tout ? Pourquoi les poids constants sont-ils aussi un problème ?

**Votre réponse :** *(à rédiger ici)*

### 4.c. La ReLU morte

Reprenons l'exercice [Quand une ReLU bloque l'apprentissage](https://openyourmath.org/exercise/LT4V) (`LT4V`) : un neurone caché ReLU puis une sortie linéaire **sans biais**, une seule donnée $(x,t)=(1,2)$, deux initialisations $A=(w,b,v)=(-1,0,1)$ et $B=(1,0,1)$. Pour reproduire ce modèle, on fige le biais de sortie à 0 (on annule son gradient).

In [ ]:
acts_relu = ["relu", "identite"]
X_un = np.array([[1.0]])
T_un = np.array([[2.0]])

historiques = {}
for nom, (w, b, v) in {"A": (-1.0, 0.0, 1.0), "B": (1.0, 0.0, 1.0)}.items():
    P = [[np.array([[w]]), np.array([b])], [np.array([[v]]), np.array([0.0])]]
    pertes = []
    for _ in range(200):
        Z, A = avant(X_un, P, acts_relu)
        pertes.append(perte(A[-1], T_un))
        grads = arriere(Z, A, P, acts_relu, T_un)
        grads[-1] = (grads[-1][0], np.zeros_like(grads[-1][1]))   # biais de sortie figé
        mise_a_jour(P, grads, 0.1)
    historiques[nom] = pertes
    print(f"initialisation {nom}: perte {pertes[0]:.3f} -> {pertes[-1]:.5f}, (w, b, v) = "
          f"({P[0][0][0, 0]:.3f}, {P[0][1][0]:.3f}, {P[1][0][0, 0]:.3f})")

assert np.isclose(historiques["A"][-1], 2.0), "depuis A, rien ne doit bouger"
assert historiques["B"][-1] < 1e-3, "depuis B, la perte doit tomber à presque 0"
plt.plot(historiques["A"], label="A (ReLU inactive)")
plt.plot(historiques["B"], label="B (ReLU active)")
plt.xlabel("itération")
plt.ylabel("perte")
plt.legend()
plt.show()

**Question.** Pourquoi la perte reste-t-elle bloquée à 2 depuis $A$, alors qu'elle n'est pas nulle ? Que se passerait-il avec plusieurs données ?

**Votre réponse :** *(à rédiger ici)*

## 5. Méthodes d'apprentissage

On passe à un problème où il faut vraiment *généraliser* : séparer deux « lunes » de points bruités. `deux_lunes` (fournie) fabrique le jeu de données ; la fonction `tracer_frontiere` colorie le plan selon la sortie du réseau.

In [ ]:
def deux_lunes(m=400, bruit=0.2, graine=0):
    rng = np.random.default_rng(graine)
    m1 = m // 2
    m0 = m - m1
    a1 = rng.uniform(0, np.pi, m1)
    a0 = rng.uniform(0, np.pi, m0)
    X1 = np.vstack([np.cos(a1), np.sin(a1)])
    X0 = np.vstack([1 - np.cos(a0), 0.5 - np.sin(a0)])
    X = np.hstack([X1, X0]) + rng.normal(0, bruit, (2, m))
    T = np.concatenate([np.ones(m1), np.zeros(m0)])[None, :]
    p = rng.permutation(m)
    return X[:, p], T[:, p]


def precision(sortie, T):
    """Proportion d'exemples bien classés (sortie >= 1/2 <=> classe 1)."""
    return float(np.mean((sortie >= 0.5) == (T >= 0.5)))


def tracer_frontiere(params, acts, X, T, ax=None, titre="", xlim=(-1.8, 2.8), ylim=(-1.4, 1.9)):
    ax = ax if ax is not None else plt.gca()
    xs = np.linspace(*xlim, 200)
    ys = np.linspace(*ylim, 200)
    XX, YY = np.meshgrid(xs, ys)
    G = np.vstack([XX.ravel(), YY.ravel()])
    Z = avant(G, params, acts)[1][-1].reshape(XX.shape)
    ax.contourf(XX, YY, Z, levels=np.linspace(0, 1, 11), cmap="RdBu", alpha=0.45)
    ax.scatter(X[0], X[1], c=np.where(T[0] >= 0.5, "tab:blue", "tab:red"), edgecolor="k", s=18, zorder=3)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_title(titre)


X_lunes, T_lunes = deux_lunes(400, 0.2, graine=0)
X_test, T_test = deux_lunes(400, 0.2, graine=1)
plt.scatter(X_lunes[0], X_lunes[1], c=np.where(T_lunes[0] >= 0.5, "tab:blue", "tab:red"), edgecolor="k", s=18)
plt.title("Deux lunes (400 points)")
plt.show()

### 5.a. Lot complet, mini-lots, un exemple à la fois

On entraîne le même réseau `[2, 16, 1]` (ReLU puis sigmoïde) pendant 300 époques de trois façons :
- **lot complet** (`taille_lot=None`), $\eta=0{,}5$ ;
- **mini-lots de 32** exemples, $\eta=0{,}5$ ;
- **un exemple à la fois** (`taille_lot=1`, descente stochastique), $\eta=0{,}05$ (un pas plus petit, car chaque pas est bruité).

**Prédiction.** Combien de mises à jour de paramètres chaque méthode fait-elle par époque (400 exemples) ? Qu'en attendez-vous pour la perte après un même nombre d'époques ? Et pour le temps de calcul ?

In [ ]:
configurations = [("lot complet", None, 0.5), ("mini-lots de 32", 32, 0.5), ("un exemple", 1, 0.05)]
acts_lunes = ["relu", "sigmoide"]
historiques = {}
for nom, taille_lot, eta in configurations:
    params = init_reseau([2, 16, 1], graine=0, echelle=0.8)
    debut = time.time()
    # À COMPLÉTER : entraîner pendant 300 époques avec ces réglages, jeu de validation (X_test, T_test)
    hist = None
    duree = time.time() - debut
    historiques[nom] = hist
    sortie = avant(X_test, params, acts_lunes)[1][-1]
    print(f"{nom:16}: perte {hist['train'][-1]:.4f} (test {hist['val'][-1]:.4f}), "
          f"précision test {precision(sortie, T_test):.3f}, durée {duree:.1f} s")
for nom, hist in historiques.items():
    plt.semilogy(hist["train"], label=nom)
plt.xlabel("époque")
plt.ylabel("perte d'entraînement")
plt.legend()
plt.show()

**Question.** Comparez les trois méthodes : nombre de mises à jour par époque, vitesse de décroissance de la perte, bruit des courbes, coût de calcul.

**Votre réponse :** *(à rédiger ici)*

### 5.b. Généraliser : entraînement et validation

Un bon réseau n'est pas celui qui s'ajuste le mieux aux points d'entraînement, mais celui qui classe bien des points **qu'il n'a jamais vus**. On construit un petit jeu d'entraînement de **30 points** très bruités, un jeu de validation de 600 points, et un grand réseau `[2, 64, 64, 1]` entraîné en lot complet pendant 2000 époques.

In [ ]:
X_ent, T_ent = deux_lunes(30, 0.35, graine=3)
X_val, T_val = deux_lunes(600, 0.35, graine=4)
acts_gros = ["relu", "relu", "sigmoide"]

params = init_reseau([2, 64, 64, 1], graine=0, echelle=0.3)
hist = entrainer(X_ent, T_ent, params, acts_gros, eta=0.3, epoques=2000, X_val=X_val, T_val=T_val)
params_fin = [[W.copy(), b.copy()] for W, b in params]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].semilogy(hist["train"], label="entraînement")
axes[0].semilogy(hist["val"], label="validation")
axes[0].set_xlabel("époque")
axes[0].set_ylabel("perte")
axes[0].legend()
tracer_frontiere(params, acts_gros, X_ent, T_ent, ax=axes[1], titre="Points d'entraînement")
tracer_frontiere(params, acts_gros, X_val[:, :150], T_val[:, :150], ax=axes[2], titre="Points de validation (150)")
plt.show()
print("précision entraînement :", precision(avant(X_ent, params, acts_gros)[1][-1], T_ent))
print("précision validation   :", precision(avant(X_val, params, acts_gros)[1][-1], T_val))

**Arrêt précoce.** Repérez l'époque où la perte de validation est minimale, puis ré-entraînez le réseau **depuis le même départ** en vous arrêtant à cette époque (la graine est la même, la trajectoire est donc identique jusque-là).

In [ ]:
epoque_min = None   # À COMPLÉTER : époque de la perte de validation minimale (np.argmin)
print("perte de validation minimale à l'époque", epoque_min)

params_arret = init_reseau([2, 64, 64, 1], graine=0, echelle=0.3)
# À COMPLÉTER : ré-entraîner jusqu'à l'époque epoque_min incluse (mêmes réglages qu'au-dessus)
...

In [ ]:
# Test
assert epoque_min is not None, "calculez epoque_min"
assert hist["val"][-1] > 1.2 * min(hist["val"]), "la perte de validation doit remonter nettement après son minimum"
perte_val_arret = perte(avant(X_val, params_arret, acts_gros)[1][-1], T_val)
perte_val_fin = perte(avant(X_val, params_fin, acts_gros)[1][-1], T_val)
assert np.isclose(perte_val_arret, min(hist["val"])), "l'arrêt précoce doit retrouver la perte de validation minimale"
print(f"perte de validation : arrêt à l'époque {epoque_min} -> {perte_val_arret:.4f} ; après 2000 époques -> {perte_val_fin:.4f}")
print("précision de validation :",
      f"arrêt précoce {precision(avant(X_val, params_arret, acts_gros)[1][-1], T_val):.3f}",
      f"| fin {precision(avant(X_val, params_fin, acts_gros)[1][-1], T_val):.3f}")

**Question.** Que montrent les deux courbes ? À quelle époque faut-il s'arrêter, et pourquoi la perte d'entraînement n'est-elle pas un bon critère ?

**Votre réponse :** *(à rédiger ici)*

### 5.c. Pénaliser les poids (régularisation $L^2$)

Dans l'exercice [Ajuster une droite en pénalisant sa pente](https://openyourmath.org/exercise/wkRy) (`wkRy`), on minimisait $J_\lambda=\text{erreur}+\lambda a^2$ et la pénalisation réduisait la pente de 1 à $\tfrac13$. On fait de même ici : on minimise $E+\lambda\sum_\ell\|W^{[\ell]}\|^2$ (les biais ne sont pas pénalisés). La dérivée de $\lambda\|W\|^2$ est $2\lambda W$ : à chaque mise à jour, le gradient de chaque matrice de poids reçoit le terme $2\lambda W^{[\ell]}$. On parle de *weight decay*, car les poids sont « tirés vers zéro » à chaque pas.

Écrivez `ajouter_decroissance(grads, params, lam)`, utilisée par `entrainer` quand `lam > 0`.

In [ ]:
def ajouter_decroissance(grads, params, lam):
    """Ajoute 2*lam*W au gradient de chaque matrice de poids W (pas aux biais)."""
    # À COMPLÉTER : renvoyer la nouvelle liste de gradients [(dW + ..., db), ...]
    raise NotImplementedError

In [ ]:
# Test
g = ajouter_decroissance([(np.zeros((2, 2)), np.zeros(2))], [[np.ones((2, 2)), np.ones(2)]], 0.1)
assert np.allclose(g[0][0], 0.2) and np.allclose(g[0][1], 0), "dW + 2 lam W pour les poids, db inchangé"

valeurs = [0.0, 1e-3, 3e-3, 1e-2, 3e-2]
perte_val, prec_val, perte_ent = [], [], []
for lam in valeurs:
    P = init_reseau([2, 64, 64, 1], graine=0, echelle=0.3)
    h = entrainer(X_ent, T_ent, P, acts_gros, eta=0.3, epoques=2000, X_val=X_val, T_val=T_val, lam=lam)
    perte_val.append(h["val"][-1])
    perte_ent.append(h["train"][-1])
    prec_val.append(precision(avant(X_val, P, acts_gros)[1][-1], T_val))
    print(f"lambda = {lam:<6}: perte entraînement {perte_ent[-1]:.4f} | perte validation {perte_val[-1]:.4f} | "
          f"précision validation {prec_val[-1]:.3f}")
assert perte_val[2] < perte_val[0], "une pénalisation modérée doit améliorer la perte de validation"

plt.semilogx(valeurs[1:], perte_val[1:], "o-", label="validation")
plt.semilogx(valeurs[1:], perte_ent[1:], "s-", label="entraînement")
plt.axhline(perte_val[0], color="gray", ls="--", label="validation sans pénalisation")
plt.xlabel("λ")
plt.ylabel("perte finale")
plt.legend()
plt.show()

**Question.** Comment évolue la perte de validation quand $\lambda$ augmente ? Que se passe-t-il pour $\lambda=0{,}03$ ? Faites le lien avec l'exercice [Ajuster une droite en pénalisant sa pente](https://openyourmath.org/exercise/wkRy) (`wkRy`).

**Votre réponse :** *(à rédiger ici)*

## 6. Pour aller plus loin : une rétropropagation automatique

Votre fonction `arriere` est écrite *pour des couches denses*. La rétropropagation est en fait la **règle de la chaîne appliquée sur un graphe de calcul**, en parcourant le graphe à l'envers. Pour le voir, on écrit une minuscule classe `Valeur` : chaque opération sur des `Valeur` mémorise comment propager un gradient vers ses opérandes.

Les méthodes `__add__` et `__pow__` sont fournies. Écrivez `__mul__` (si $s=u\,v$ alors $\partial s/\partial u=v$ et $\partial s/\partial v=u$), `relu`, et `backward`, qui parcourt le graphe dans l'ordre inverse.

In [ ]:
class Valeur:
    def __init__(self, data, enfants=()):
        self.data = float(data)
        self.grad = 0.0
        self._enfants = enfants
        self._arriere = lambda: None

    def __add__(self, autre):
        autre = autre if isinstance(autre, Valeur) else Valeur(autre)
        sortie = Valeur(self.data + autre.data, (self, autre))

        def _arriere():
            self.grad += sortie.grad
            autre.grad += sortie.grad
        sortie._arriere = _arriere
        return sortie

    def __mul__(self, autre):
        autre = autre if isinstance(autre, Valeur) else Valeur(autre)
        sortie = Valeur(self.data * autre.data, (self, autre))
        # À COMPLÉTER : définir _arriere (ajouter aux grad des deux opérandes) et l'attacher à sortie
        return sortie

    def __pow__(self, p):
        sortie = Valeur(self.data ** p, (self,))

        def _arriere():
            self.grad += p * self.data ** (p - 1) * sortie.grad
        sortie._arriere = _arriere
        return sortie

    def relu(self):
        sortie = Valeur(max(0.0, self.data), (self,))
        # À COMPLÉTER : définir _arriere (dérivée de ReLU : 1 si data > 0, sinon 0) et l'attacher à sortie
        return sortie

    def backward(self):
        """Calcule le gradient de self par rapport à tous les noeuds du graphe."""
        ordre, vus = [], set()

        def parcourir(v):
            if id(v) not in vus:
                vus.add(id(v))
                for e in v._enfants:
                    parcourir(e)
                ordre.append(v)
        parcourir(self)
        self.grad = 1.0
        # À COMPLÉTER : appeler _arriere de chaque noeud, des sorties vers les entrées
        raise NotImplementedError

    __radd__ = __add__
    __rmul__ = __mul__

    def __neg__(self):
        return self * -1

    def __sub__(self, autre):
        return self + (-autre)

**Test : encore l'exercice *Une rétropropagation de bout en bout*.** On écrit le calcul tel quel, puis on appelle `backward` : on doit retrouver les gradients $(2,2,1,1)$ sans avoir écrit une seule formule de rétropropagation.

In [ ]:
# Test
w1, b1, w2, b2 = Valeur(1.0), Valeur(0.0), Valeur(2.0), Valeur(0.0)
x, t = 1.0, 1.0
h = (w1 * x + b1).relu()
F = w2 * h + b2
E = 0.5 * (F - t) ** 2
E.backward()
assert np.isclose(E.data, 0.5)
assert np.allclose([w1.grad, b1.grad, w2.grad, b2.grad], [2.0, 2.0, 1.0, 1.0]), (w1.grad, b1.grad, w2.grad, b2.grad)
print("Rétropropagation automatique : gradients", (w1.grad, b1.grad, w2.grad, b2.grad))

**Question.** Pourquoi retrouve-t-on les mêmes gradients que par la rétropropagation écrite à la main ?

**Votre réponse :** *(à rédiger ici)*

## Fiche de résultats

Cette cellule recalcule les valeurs demandées dans l'exercice [TP Python : rétropropagation et méthodes d'apprentissage](https://openyourmath.org/exercise/bNCl) (`bNCl`) (questions 1, 2, 4, 5, 7 et 8). Elle ne fonctionne que si **toutes** les cellules précédentes sont complétées et correctes, et elle prend une quinzaine de secondes. Recopiez ce qu'elle affiche : toutes les graines sont fixées, vos résultats doivent être exactement ceux attendus.

In [ ]:
# Fiche de résultats
RESULTATS = {}

# Q1 : vérification des gradients
rng_f = np.random.default_rng(1)
P_f = init_reseau([3, 4, 2], graine=2, echelle=1.0)
X_f = rng_f.normal(size=(3, 6))
T_f = rng_f.uniform(size=(2, 6))
ecart = verifier_gradient(P_f, ["sigmoide", "sigmoide"], X_f, T_f)
RESULTATS["q1"] = float(ecart)
print(f"Q1. Écart relatif maximal (réseau [3, 4, 2] sigmoïde) : {ecart:.1e}")

# Q2 : le XOR
P_f = init_reseau([2, 3, 1], graine=0, echelle=1.0)
h_f = entrainer(X_xor, T_xor, P_f, acts_xor, eta=2.0, epoques=3000)
sorties = np.round(avant(X_xor, P_f, acts_xor)[1][-1][0], 2)
RESULTATS["q2"] = {"perte": float(h_f["train"][-1]), "sorties": [float(v) for v in sorties]}
print(f"Q2. XOR [2, 3, 1], eta = 2, 3000 époques : perte finale {h_f['train'][-1]:.4f}, sorties {sorties}")

# Q4 : la ReLU morte (biais de sortie figé)
def relu_morte(w, b, v, iterations=200, eta=0.1):
    P = [[np.array([[w]]), np.array([b])], [np.array([[v]]), np.array([0.0])]]
    for _ in range(iterations):
        Z, A = avant(np.array([[1.0]]), P, ["relu", "identite"])
        g = arriere(Z, A, P, ["relu", "identite"], np.array([[2.0]]))
        g[-1] = (g[-1][0], np.zeros_like(g[-1][1]))
        mise_a_jour(P, g, eta)
    return P, perte(avant(np.array([[1.0]]), P, ["relu", "identite"])[1][-1], np.array([[2.0]]))


P_A, perte_A = relu_morte(-1.0, 0.0, 1.0)
P_B, perte_B = relu_morte(1.0, 0.0, 1.0)
RESULTATS["q4"] = {"perte_A": float(perte_A), "perte_B": float(perte_B), "w_B": float(P_B[0][0][0, 0])}
print(f"Q4. ReLU morte, après 200 itérations : perte {perte_A:.3f} depuis A, {perte_B:.3f} depuis B ; w = {P_B[0][0][0, 0]:.2f} depuis B")

# Q5 : lot complet, mini-lots, un exemple à la fois
RESULTATS["q5"] = {}
for nom, taille_lot, eta in [("lot complet", None, 0.5), ("mini-lots de 32", 32, 0.5), ("un exemple", 1, 0.05)]:
    P = init_reseau([2, 16, 1], graine=0, echelle=0.8)
    h = entrainer(X_lunes, T_lunes, P, acts_lunes, eta=eta, epoques=300, taille_lot=taille_lot)
    pr = precision(avant(X_test, P, acts_lunes)[1][-1], T_test)
    RESULTATS["q5"][nom] = {"perte": float(h["train"][-1]), "precision": float(pr)}
    print(f"Q5. {nom:16}: perte d'entraînement {h['train'][-1]:.4f}, précision de test {pr:.3f}")

# Q7 : arrêt précoce
P_gros = init_reseau([2, 64, 64, 1], graine=0, echelle=0.3)
h_gros = entrainer(X_ent, T_ent, P_gros, acts_gros, eta=0.3, epoques=2000, X_val=X_val, T_val=T_val)
k_min = int(np.argmin(h_gros["val"]))
P_es = init_reseau([2, 64, 64, 1], graine=0, echelle=0.3)
entrainer(X_ent, T_ent, P_es, acts_gros, eta=0.3, epoques=k_min + 1)
prec_fin = precision(avant(X_val, P_gros, acts_gros)[1][-1], T_val)
prec_es = precision(avant(X_val, P_es, acts_gros)[1][-1], T_val)
RESULTATS["q7"] = {"epoque": k_min, "val_min": float(min(h_gros["val"])), "val_fin": float(h_gros["val"][-1]),
                   "precision_arret": float(prec_es), "precision_fin": float(prec_fin)}
print(f"Q7. Perte de validation minimale à l'époque {k_min} ({min(h_gros['val']):.4f}) contre {h_gros['val'][-1]:.4f} à la fin ; "
      f"précision de validation {prec_es:.3f} (arrêt précoce) contre {prec_fin:.3f} (fin)")

# Q8 : pénalisation L2
RESULTATS["q8"] = {}
for lam in [0.0, 1e-3, 3e-3, 1e-2, 3e-2]:
    P = init_reseau([2, 64, 64, 1], graine=0, echelle=0.3)
    h = entrainer(X_ent, T_ent, P, acts_gros, eta=0.3, epoques=2000, X_val=X_val, T_val=T_val, lam=lam)
    pr = precision(avant(X_val, P, acts_gros)[1][-1], T_val)
    RESULTATS["q8"][str(lam)] = {"perte_val": float(h["val"][-1]), "precision": float(pr)}
    print(f"Q8. lambda = {lam:<6}: perte de validation {h['val'][-1]:.4f}, précision de validation {pr:.3f}")

## Bilan

Vous avez programmé de bout en bout un réseau de neurones : propagation avant, perte, **rétropropagation** (vérifiée sur un exemple calculé à la main et par différences finies), mise à jour, puis les principales méthodes d'apprentissage : taille de lot, pas, initialisation, validation, arrêt précoce, pénalisation. Chaque outil se justifie par un problème concret observé : plateau, symétrie, neurone mort, sur-apprentissage.